# dispatch-back-fn-from-recipe — worked example 3: Dispatch back functions for a ternary forward function with three parents

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dispatch-back-fn-from-recipe`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The dispatch pattern scales naturally to functions with any number of inputs. For a ternary function (three arguments), `recipe.parents` has three entries keyed by argnums 0, 1, and 2. The dispatch loop emits three triples. Each back function receives the output gradient and can use the saved recipe args to compute the gradient for its specific input.

## Worked solution

Function `weighted_sum(a, b, c) = 0.5*a + 0.3*b + 0.2*c`. Gradients:
- argnum 0: `back_0(grad) = 0.5 * grad`
- argnum 1: `back_1(grad) = 0.3 * grad`
- argnum 2: `back_2(grad) = 0.2 * grad`

Node `out` has `parents = {0: a, 1: b, 2: c}`. The dispatch loop yields:
1. `(0, a, back_0)` — for gradient flowing to `a`.
2. `(1, b, back_1)` — for gradient flowing to `b`.
3. `(2, c, back_2)` — for gradient flowing to `c`.

Each triple carries everything the backward caller needs: which argnum it is, which parent receives the gradient, and how to transform the output gradient into the input gradient.

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Recipe:
    func: Callable
    parents: dict

class FakeTensor:
    def __init__(self, name, recipe=None):
        self.name = name
        self.recipe = recipe
    def __repr__(self): return f'FakeTensor({self.name})'

def weighted_sum(a, b, c): return 0.5*a + 0.3*b + 0.2*c

# Scale factors match the coefficients
back_0 = lambda grad, out, a, b, c: 0.5 * grad
back_1 = lambda grad, out, a, b, c: 0.3 * grad
back_2 = lambda grad, out, a, b, c: 0.2 * grad
back_0.__name__ = 'back_0'
back_1.__name__ = 'back_1'
back_2.__name__ = 'back_2'

a, b, c = FakeTensor('a'), FakeTensor('b'), FakeTensor('c')
out = FakeTensor('out', Recipe(func=weighted_sum, parents={0: a, 1: b, 2: c}))

back_funcs = {
    (weighted_sum, 0): back_0,
    (weighted_sum, 1): back_1,
    (weighted_sum, 2): back_2,
}

def dispatch_back_fns(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    return results

triples = dispatch_back_fns(out, back_funcs)
print(f'{len(triples)} triples dispatched')
grad_out = 10.0
for argnum, parent, fn in sorted(triples, key=lambda t: t[0]):
    g = fn(grad_out, out, a, b, c)
    print(f'  argnum={argnum} parent={parent.name} grad={g:.4f}')

assert len(triples) == 3
expected_grads = {0: 5.0, 1: 3.0, 2: 2.0}
for argnum, parent, fn in triples:
    assert abs(fn(grad_out, out, a, b, c) - expected_grads[argnum]) < 1e-6
print('All assertions passed.')